In [6]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

url = "https://www.snopes.com/fact-check/?filter=mixture"
response = requests.get(url, headers=HEADERS, timeout=15)
soup = BeautifulSoup(response.text, "html.parser")

# Show all unique div classes containing 'article'
print("--- DIV[article] CLASSES ---")
divs = soup.select("div[class*='article']")
seen = set()
for d in divs:
    cls = " ".join(d.get("class", []))
    if cls not in seen:
        seen.add(cls)
        print("  CLASS:", cls)
        print("  TEXT: ", d.get_text(strip=True)[:100])
        print()

# Show fact-check links with context
print("--- FACT CHECK LINKS ---")
links = soup.select("a[href*='/fact-check/']")
for a in links[:5]:
    print("  URL: ", a['href'])
    print("  TEXT:", a.get_text(strip=True)[:100])
    parent = a.find_parent()
    print("  PARENT:", parent.name, parent.get('class', []))
    print()

--- DIV[article] CLASSES ---
  CLASS: article_list_cont
  TEXT:  Have Social Security checks been renamed 'federal benefit payments'? Here's the bottom lineWritten b

  CLASS: article_wrapper
  TEXT:  Have Social Security checks been renamed 'federal benefit payments'? Here's the bottom lineWritten b

  CLASS: outer_article_img
  TEXT:  

  CLASS: article_img
  TEXT:  

  CLASS: article_text
  TEXT:  Have Social Security checks been renamed 'federal benefit payments'? Here's the bottom lineWritten b

  CLASS: article_info_wrap
  TEXT:  Have Social Security checks been renamed 'federal benefit payments'? Here's the bottom lineWritten b

--- FACT CHECK LINKS ---
  URL:  /fact-check/
  TEXT: Fact Checks
  PARENT: div ['category_row']

  URL:  https://www.snopes.com/fact-check/social-security-benefit-payments-name/
  TEXT: Have Social Security checks been renamed 'federal benefit payments'? Here's the bottom lineWritten b
  PARENT: div ['article_wrapper']

  URL:  https://www.snopes.com/fa

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

def scrape_snopes_mixture(max_pages=50):
    base_url = "https://www.snopes.com/fact-check/?filter=mixture"
    results = []

    for page in range(1, max_pages + 1):
        url = f"{base_url}&pagenum={page}"
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
        except Exception as e:
            print(f"Request failed page {page}: {e}")
            break

        soup = BeautifulSoup(response.text, "html.parser")
        cards = soup.select("div.article_wrapper")

        if not cards:
            print(f"No more articles at page {page}, stopping.")
            break

        for card in cards:
            link_tag = card.select_one("a[href*='/fact-check/']")
            title_tag = card.select_one("div.article_text")

            if link_tag and title_tag:
                href = link_tag.get("href", "")
                if href.startswith("/"):
                    href = "https://www.snopes.com" + href

                title = title_tag.get_text(strip=True)

                # Fetch full article text
                text = scrape_article_text(href)

                results.append({
                    "title": title,
                    "text": text,
                    "label": "misleading"
                })
                print(f"  [{len(results)}] {title[:70]}")
                time.sleep(random.uniform(1.0, 2.0))

        print(f"Page {page} done -- {len(results)} total so far")
        time.sleep(random.uniform(1.5, 3.0))

    return results


def scrape_article_text(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")

        body = (
            soup.select_one("div.single-body") or
            soup.select_one("div.article-text") or
            soup.select_one("div[class*='body']") or
            soup.select_one("main")
        )
        return body.get_text(separator=" ", strip=True) if body else ""

    except Exception as e:
        print(f"  Failed: {url} -- {e}")
        return ""


def build_snopes_dataset(max_pages=50):
    print("Scraping Snopes mixture articles...\n")
    results = scrape_snopes_mixture(max_pages)

    df = pd.DataFrame(results)[["title", "text", "label"]]  # only 3 columns
    df = df[df["text"].str.len() > 50].reset_index(drop=True)

    df.to_csv("snopes_misleading.csv", index=False)
    print(f"\nDone! Saved {len(df)} rows to snopes_misleading.csv")
    print(df.head())
    return df


# RUN
df = build_snopes_dataset(max_pages=50)

Scraping Snopes mixture articles...

  [1] Have Social Security checks been renamed 'federal benefit payments'? H
  [2] Video doesn't show downed US pilot in Iranian custodyWritten by:Jordan
  [3] White House posted and deleted video of Trump mocking SCOTUS justices,
  [4] Robert Morris, former Trump faith adviser, released on probation after
  [5] Did RFK Jr. announce Nutella is being pulled from US shelves?Written b
  [6] Is Trump renaming Washington Monument after himself, plating it in gol
  [7] Fake image of dad wearing MAGA hat blocking trans person from women's 
  [8] Did Iran say it's starting to achieve US regime change following Pam B
  [9] Image of Trump, Epstein with young female and children is fake. Here's
  [10] Kamala Harris never said, 'Iran is a country, but we don't live there,
  [11] Did Oprah try to bribe John Fetterman to vote against Trump's 'Big Bea
  [12] Why Trump's birthright citizenship executive order wouldn't impact his
  [13] Kurt Russell didn't receive K

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

# ── HELPER ─────────────────────────────────────────────────────────
def safe_get(url, timeout=15):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        return r
    except Exception as e:
        print(f"  Request failed: {url} -- {e}")
        return None


# ── SOURCE 1: SNOPES (Mixture) ──────────────────────────────────────
def scrape_snopes(max_pages=50):
    results = []
    base_url = "https://www.snopes.com/fact-check/?filter=mixture"

    for page in range(1, max_pages + 1):
        r = safe_get(f"{base_url}&pagenum={page}")
        if not r:
            break

        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("div.article_wrapper")
        if not cards:
            print(f"[Snopes] No more articles at page {page}")
            break

        for card in cards:
            link_tag = card.select_one("a[href*='/fact-check/']")
            title_tag = card.select_one("div.article_text")
            if not link_tag or not title_tag:
                continue

            href = link_tag.get("href", "")
            if href.startswith("/"):
                href = "https://www.snopes.com" + href

            title = title_tag.get_text(strip=True)
            text  = scrape_snopes_article(href)

            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [Snopes {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))

        print(f"[Snopes] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))

    return results


def scrape_snopes_article(url):
    r = safe_get(url)
    if not r:
        return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = (
        soup.select_one("div.single-body") or
        soup.select_one("div.article-text") or
        soup.select_one("div[class*='body']") or
        soup.select_one("main")
    )
    return body.get_text(separator=" ", strip=True) if body else ""


# ── SOURCE 2: POLITIFACT (Half True + Mostly False) ─────────────────
def scrape_politifact(max_pages=30):
    results = []
    ratings = ["half-true", "mostly-false"]

    for rating in ratings:
        for page in range(1, max_pages + 1):
            url = f"https://www.politifact.com/factchecks/list/?ruling={rating}&page={page}"
            r = safe_get(url)
            if not r:
                break

            soup = BeautifulSoup(r.text, "html.parser")
            items = soup.select("li.o-listicle__item")
            if not items:
                print(f"[PolitiFact-{rating}] No more items at page {page}")
                break

            for item in items:
                link_tag = item.select_one("a.m-statement__quote")
                if not link_tag:
                    continue

                title = link_tag.get_text(strip=True)
                href  = "https://www.politifact.com" + link_tag["href"]
                text  = scrape_politifact_article(href)

                if text:
                    results.append({"title": title, "text": text, "label": "misleading"})
                    print(f"  [PolitiFact {len(results)}] {title[:70]}")
                time.sleep(random.uniform(1.0, 2.0))

            print(f"[PolitiFact-{rating}] Page {page} done -- {len(results)} total")
            time.sleep(random.uniform(1.5, 2.5))

    return results


def scrape_politifact_article(url):
    r = safe_get(url)
    if not r:
        return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = soup.select_one("article.m-textblock")
    return body.get_text(separator=" ", strip=True) if body else ""


# ── SOURCE 3: FACTCHECK.ORG ─────────────────────────────────────────
def scrape_factcheck(max_pages=20):
    results = []

    for page in range(1, max_pages + 1):
        url = f"https://www.factcheck.org/fake-news/?paged={page}"
        r = safe_get(url)
        if not r:
            break

        soup = BeautifulSoup(r.text, "html.parser")
        articles = soup.select("article")
        if not articles:
            print(f"[FactCheck] No more articles at page {page}")
            break

        for art in articles:
            link_tag = art.select_one("a")
            title_tag = art.select_one("h2, h3")
            if not link_tag or not title_tag:
                continue

            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            text  = scrape_factcheck_article(href)

            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [FactCheck {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))

        print(f"[FactCheck] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))

    return results


def scrape_factcheck_article(url):
    r = safe_get(url)
    if not r:
        return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = soup.select_one("div.entry-content")
    return body.get_text(separator=" ", strip=True) if body else ""


# ── DEDUPLICATE ─────────────────────────────────────────────────────
def deduplicate(df):
    before = len(df)

    df = df.drop_duplicates(subset=["title"])
    df = df.drop_duplicates(subset=["text"])
    df = df[df["text"].str.len() > 100]

    df["title_norm"] = df["title"].str.lower().str.strip()
    df = df.drop_duplicates(subset=["title_norm"])
    df = df.drop(columns=["title_norm"])

    after = len(df)
    print(f"Deduplication: {before} -> {after} rows ({before - after} removed)")
    return df.reset_index(drop=True)


# ── MAIN BUILD FUNCTION ─────────────────────────────────────────────
def build_full_dataset(existing_csv=None, max_pages=50):
    all_results = []

    print("\n" + "="*50)
    print("SCRAPING SNOPES...")
    print("="*50)
    all_results += scrape_snopes(max_pages=max_pages)

    print("\n" + "="*50)
    print("SCRAPING POLITIFACT...")
    print("="*50)
    all_results += scrape_politifact(max_pages=30)

    print("\n" + "="*50)
    print("SCRAPING FACTCHECK.ORG...")
    print("="*50)
    all_results += scrape_factcheck(max_pages=20)

    # Build new dataframe
    new_df = pd.DataFrame(all_results)[["title", "text", "label"]]
    print(f"\nTotal freshly scraped: {len(new_df)} articles")

    # Merge with existing Snopes CSV
    if existing_csv:
        print(f"\nLoading existing dataset: {existing_csv}")
        existing_df = pd.read_csv(existing_csv)[["title", "text", "label"]]
        print(f"Existing rows: {len(existing_df)}")
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        print(f"Combined before dedup: {len(combined_df)}")
    else:
        combined_df = new_df

    # Deduplicate
    print("\nDeduplicating...")
    final_df = deduplicate(combined_df)

    # Save
    final_df.to_csv("misleading_all_sources.csv", index=False)
    print(f"\nDone! Saved {len(final_df)} rows to misleading_all_sources.csv")
    print(f"\nLabel distribution:\n{final_df['label'].value_counts()}")
    print("\nSample rows:")
    print(final_df[["title", "label"]].head())
    return final_df


# ── RUN ─────────────────────────────────────────────────────────────
df = build_full_dataset(existing_csv="snopes_misleading.csv", max_pages=50)


SCRAPING SNOPES...
  [Snopes 1] Have Social Security checks been renamed 'federal benefit payments'? H
  [Snopes 2] Video doesn't show downed US pilot in Iranian custodyWritten by:Jordan
  [Snopes 3] White House posted and deleted video of Trump mocking SCOTUS justices,
  [Snopes 4] Robert Morris, former Trump faith adviser, released on probation after
  [Snopes 5] Did RFK Jr. announce Nutella is being pulled from US shelves?Written b
  [Snopes 6] Is Trump renaming Washington Monument after himself, plating it in gol
  [Snopes 7] Fake image of dad wearing MAGA hat blocking trans person from women's 
  [Snopes 8] Did Iran say it's starting to achieve US regime change following Pam B
  [Snopes 9] Image of Trump, Epstein with young female and children is fake. Here's
  [Snopes 10] Kamala Harris never said, 'Iran is a country, but we don't live there,
  [Snopes 11] Did Oprah try to bribe John Fetterman to vote against Trump's 'Big Bea
  [Snopes 12] Why Trump's birthright citizenship execu

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

# ── HELPER ──────────────────────────────────────────────────────────
def safe_get(url, timeout=15):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        return r
    except Exception as e:
        print(f"  Request failed: {url} -- {e}")
        return None

# ── SOURCE 1: SNOPES ────────────────────────────────────────────────
def scrape_snopes(max_pages=50):
    results = []
    base_url = "https://www.snopes.com/fact-check/?filter=mixture"
    for page in range(1, max_pages + 1):
        r = safe_get(f"{base_url}&pagenum={page}")
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("div.article_wrapper")
        if not cards:
            print(f"[Snopes] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a[href*='/fact-check/']")
            title_tag = card.select_one("div.article_text")
            if not link_tag or not title_tag: continue
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://www.snopes.com" + href
            title = title_tag.get_text(strip=True)
            text = scrape_snopes_article(href)
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [Snopes {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))
        print(f"[Snopes] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))
    return results

def scrape_snopes_article(url):
    r = safe_get(url)
    if not r: return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = (soup.select_one("div.single-body") or
            soup.select_one("div.article-text") or
            soup.select_one("div[class*='body']") or
            soup.select_one("main"))
    return body.get_text(separator=" ", strip=True) if body else ""

# ── SOURCE 2: POLITIFACT ────────────────────────────────────────────
def scrape_politifact(max_pages=30):
    results = []
    ratings = ["half-true", "mostly-false"]
    for rating in ratings:
        for page in range(1, max_pages + 1):
            url = f"https://www.politifact.com/factchecks/list/?ruling={rating}&page={page}"
            r = safe_get(url)
            if not r: break
            soup = BeautifulSoup(r.text, "html.parser")
            items = soup.select("li.o-listicle__item")
            if not items:
                print(f"[PolitiFact-{rating}] Stopped at page {page}")
                break
            for item in items:
                link_tag = item.select_one("a.m-statement__quote")
                if not link_tag: continue
                title = link_tag.get_text(strip=True)
                href = "https://www.politifact.com" + link_tag["href"]
                text = scrape_politifact_article(href)
                if text:
                    results.append({"title": title, "text": text, "label": "misleading"})
                    print(f"  [PolitiFact {len(results)}] {title[:70]}")
                time.sleep(random.uniform(1.0, 2.0))
            print(f"[PolitiFact-{rating}] Page {page} done -- {len(results)} total")
            time.sleep(random.uniform(1.5, 2.5))
    return results

def scrape_politifact_article(url):
    r = safe_get(url)
    if not r: return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = soup.select_one("article.m-textblock")
    return body.get_text(separator=" ", strip=True) if body else ""

# ── SOURCE 3: FACTCHECK.ORG ─────────────────────────────────────────
def scrape_factcheck(max_pages=20):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://www.factcheck.org/fake-news/?paged={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        articles = soup.select("article")
        if not articles:
            print(f"[FactCheck] Stopped at page {page}")
            break
        for art in articles:
            link_tag = art.select_one("a")
            title_tag = art.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            text = scrape_factcheck_article(href)
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [FactCheck {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))
        print(f"[FactCheck] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))
    return results

def scrape_factcheck_article(url):
    r = safe_get(url)
    if not r: return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = soup.select_one("div.entry-content")
    return body.get_text(separator=" ", strip=True) if body else ""

# ── SOURCE 4: AFP FACT CHECK ────────────────────────────────────────
def scrape_afp(max_pages=20):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://factcheck.afp.com/list/misleading?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article.article-card")
        if not cards:
            print(f"[AFP] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3, .article-title")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://factcheck.afp.com" + href
            text = scrape_afp_article(href)
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [AFP {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))
        print(f"[AFP] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))
    return results

def scrape_afp_article(url):
    r = safe_get(url)
    if not r: return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = (soup.select_one("div.article-body") or
            soup.select_one("div.article-content") or
            soup.select_one("main"))
    return body.get_text(separator=" ", strip=True) if body else ""

# ── SOURCE 5: FULLFACT.ORG ──────────────────────────────────────────
def scrape_fullfact(max_pages=20):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://fullfact.org/latest/?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article")
        if not cards:
            print(f"[FullFact] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://fullfact.org" + href
            text = scrape_fullfact_article(href)
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [FullFact {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))
        print(f"[FullFact] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))
    return results

def scrape_fullfact_article(url):
    r = safe_get(url)
    if not r: return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = (soup.select_one("div.article-content") or
            soup.select_one("div.entry-content") or
            soup.select_one("main"))
    return body.get_text(separator=" ", strip=True) if body else ""

# ── SOURCE 6: LEAD STORIES ──────────────────────────────────────────
def scrape_leadstories(max_pages=20):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://leadstories.com/hoax-alert/?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article")
        if not cards:
            print(f"[LeadStories] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://leadstories.com" + href
            text = scrape_leadstories_article(href)
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [LeadStories {len(results)}] {title[:70]}")
            time.sleep(random.uniform(1.0, 2.0))
        print(f"[LeadStories] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.5, 2.5))
    return results

def scrape_leadstories_article(url):
    r = safe_get(url)
    if not r: return ""
    soup = BeautifulSoup(r.text, "html.parser")
    body = (soup.select_one("div.entry-content") or
            soup.select_one("div.article-body") or
            soup.select_one("main"))
    return body.get_text(separator=" ", strip=True) if body else ""

# ── SMART DEDUPLICATE ───────────────────────────────────────────────
def deduplicate(df):
    before = len(df)
    df = df.drop_duplicates(subset=["title"])
    df = df.drop_duplicates(subset=["text"])
    df = df[df["text"].str.len() > 100]
    df["title_norm"] = df["title"].str.lower().str.strip()
    df = df.drop_duplicates(subset=["title_norm"])
    df = df.drop(columns=["title_norm"])
    df = df.reset_index(drop=True)
    after = len(df)
    print(f"Deduplication: {before} -> {after} rows ({before - after} removed)")
    return df

# ── MAIN ────────────────────────────────────────────────────────────
def build_full_dataset(existing_csv=None, max_pages=50):
    all_results = []

    sources = [
        ("SNOPES",       scrape_snopes,       {"max_pages": max_pages}),
        ("POLITIFACT",   scrape_politifact,   {"max_pages": 30}),
        ("FACTCHECK",    scrape_factcheck,    {"max_pages": 20}),
        ("AFP",          scrape_afp,          {"max_pages": 20}),
        ("FULLFACT",     scrape_fullfact,     {"max_pages": 20}),
        ("LEADSTORIES",  scrape_leadstories,  {"max_pages": 20}),
    ]

    for name, func, kwargs in sources:
        print(f"\n{'='*50}")
        print(f"SCRAPING {name}...")
        print(f"{'='*50}")
        try:
            data = func(**kwargs)
            all_results += data
            print(f"{name} total: {len(data)} articles")
        except Exception as e:
            print(f"{name} failed completely: {e}")

    new_df = pd.DataFrame(all_results)[["title", "text", "label"]]
    print(f"\nTotal freshly scraped: {len(new_df)}")

    # Merge with existing CSV
    if existing_csv and os.path.exists(existing_csv):
        print(f"\nLoading existing: {existing_csv}")
        existing_df = pd.read_csv(existing_csv)[["title", "text", "label"]]
        print(f"Existing rows: {len(existing_df)}")
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
        print(f"Combined before dedup: {len(combined_df)}")
    else:
        combined_df = new_df

    print("\nDeduplicating...")
    final_df = deduplicate(combined_df)

    final_df.to_csv("misleading_all_sources.csv", index=False)
    print(f"\nDone! Saved {len(final_df)} rows to misleading_all_sources.csv")
    print(f"\nLabel distribution:\n{final_df['label'].value_counts()}")
    print("\nSample:")
    print(final_df[["title", "label"]].head())
    return final_df

# ── RUN ─────────────────────────────────────────────────────────────
df = build_full_dataset(existing_csv="snopes_misleading.csv", max_pages=50)


SCRAPING SNOPES...
  [Snopes 1] Have Social Security checks been renamed 'federal benefit payments'? H
  [Snopes 2] Video doesn't show downed US pilot in Iranian custodyWritten by:Jordan
  [Snopes 3] White House posted and deleted video of Trump mocking SCOTUS justices,
  [Snopes 4] Robert Morris, former Trump faith adviser, released on probation after
  [Snopes 5] Did RFK Jr. announce Nutella is being pulled from US shelves?Written b
  [Snopes 6] Is Trump renaming Washington Monument after himself, plating it in gol
  [Snopes 7] Fake image of dad wearing MAGA hat blocking trans person from women's 
  [Snopes 8] Did Iran say it's starting to achieve US regime change following Pam B
  [Snopes 9] Image of Trump, Epstein with young female and children is fake. Here's
  [Snopes 10] Kamala Harris never said, 'Iran is a country, but we don't live there,
  [Snopes 11] Did Oprah try to bribe John Fetterman to vote against Trump's 'Big Bea
  [Snopes 12] Why Trump's birthright citizenship execu

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

# ── HELPER ──────────────────────────────────────────────────────────
def safe_get(url, timeout=15):
    try:
        r = requests.get(url, headers=HEADERS, timeout=timeout)
        return r
    except Exception as e:
        print(f"  Request failed: {url} -- {e}")
        return None

def truncate_60_words(text):
    words = text.split()
    return " ".join(words[:60])

def extract_text(soup, selectors):
    for sel in selectors:
        body = soup.select_one(sel)
        if body:
            return truncate_60_words(body.get_text(separator=" ", strip=True))
    return ""

# ── SOURCE 1: SNOPES (Mixture) ~1500 ────────────────────────────────
def scrape_snopes(max_pages=150):
    results = []
    base_url = "https://www.snopes.com/fact-check/?filter=mixture"
    for page in range(1, max_pages + 1):
        r = safe_get(f"{base_url}&pagenum={page}")
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("div.article_wrapper")
        if not cards:
            print(f"[Snopes] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a[href*='/fact-check/']")
            title_tag = card.select_one("div.article_text")
            if not link_tag or not title_tag: continue
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://www.snopes.com" + href
            title = title_tag.get_text(strip=True)
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.single-body", "div.article-text", "div[class*='body']", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [Snopes {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[Snopes] Page {page} done -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 2: POLITIFACT Half-True ~1000 ────────────────────────────
def scrape_politifact(max_pages=100):
    results = []
    ratings = ["half-true", "mostly-false", "barely-true"]
    for rating in ratings:
        for page in range(1, max_pages + 1):
            url = f"https://www.politifact.com/factchecks/list/?ruling={rating}&page={page}"
            r = safe_get(url)
            if not r: break
            soup = BeautifulSoup(r.text, "html.parser")
            items = soup.select("li.o-listicle__item")
            if not items:
                print(f"[PolitiFact-{rating}] Stopped at page {page}")
                break
            for item in items:
                link_tag = item.select_one("a.m-statement__quote")
                if not link_tag: continue
                title = link_tag.get_text(strip=True)
                href = "https://www.politifact.com" + link_tag["href"]
                r2 = safe_get(href)
                if not r2: continue
                s2 = BeautifulSoup(r2.text, "html.parser")
                text = extract_text(s2, ["article.m-textblock", "div.m-textblock", "main"])
                if text:
                    results.append({"title": title, "text": text, "label": "misleading"})
                    print(f"  [PolitiFact {len(results)}] {title[:70]}")
                time.sleep(random.uniform(0.8, 1.5))
            print(f"[PolitiFact-{rating}] Page {page} -- {len(results)} total")
            time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 3: FACTCHECK.ORG ~500 ────────────────────────────────────
def scrape_factcheck(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://www.factcheck.org/fake-news/?paged={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        articles = soup.select("article")
        if not articles:
            print(f"[FactCheck] Stopped at page {page}")
            break
        for art in articles:
            link_tag = art.select_one("a")
            title_tag = art.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.entry-content", "div.article-body", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [FactCheck {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[FactCheck] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 4: AFP FACT CHECK ~800 ───────────────────────────────────
def scrape_afp(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://factcheck.afp.com/list/misleading?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article.article-card")
        if not cards:
            print(f"[AFP] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3, .article-title")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://factcheck.afp.com" + href
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.article-body", "div.article-content", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [AFP {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[AFP] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 5: FULLFACT ~600 ─────────────────────────────────────────
def scrape_fullfact(max_pages=60):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://fullfact.org/latest/?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article")
        if not cards:
            print(f"[FullFact] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://fullfact.org" + href
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.article-content", "div.entry-content", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [FullFact {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[FullFact] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 6: LEAD STORIES ~700 ─────────────────────────────────────
def scrape_leadstories(max_pages=70):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://leadstories.com/hoax-alert/?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article")
        if not cards:
            print(f"[LeadStories] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://leadstories.com" + href
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.entry-content", "div.article-body", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [LeadStories {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[LeadStories] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 7: VISHVASNEWS ~500 ──────────────────────────────────────
def scrape_vishvas(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://www.vishvasnews.com/english/fact-check/page/{page}/"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article")
        if not cards:
            print(f"[Vishvas] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.post-content", "div.entry-content", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [Vishvas {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[Vishvas] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 8: BOOMLIVE ~500 ─────────────────────────────────────────
def scrape_boomlive(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://www.boomlive.in/fact-check?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div.story-card, div.card")
        if not cards:
            print(f"[BoomLive] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3, .headline")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://www.boomlive.in" + href
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.article-body", "div.entry-content", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [BoomLive {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[BoomLive] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 9: CHECKYOURFACT ~400 ────────────────────────────────────
def scrape_checkyourfact(max_pages=40):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://checkyourfact.com/page/{page}/"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article")
        if not cards:
            print(f"[CheckYourFact] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.entry-content", "div.article-body", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [CheckYourFact {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[CheckYourFact] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── SOURCE 10: LOGICALLY FACTS ~400 ─────────────────────────────────
def scrape_logically(max_pages=40):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://www.logicallyfacts.com/en/fact-checks?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div.fact-check-card, div.card")
        if not cards:
            print(f"[Logically] Stopped at page {page}")
            break
        for card in cards:
            link_tag = card.select_one("a")
            title_tag = card.select_one("h2, h3, .title")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href = link_tag.get("href", "")
            if href.startswith("/"): href = "https://www.logicallyfacts.com" + href
            r2 = safe_get(href)
            if not r2: continue
            s2 = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.article-body", "div.content", "main"])
            if text:
                results.append({"title": title, "text": text, "label": "misleading"})
                print(f"  [Logically {len(results)}] {title[:70]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[Logically] Page {page} -- {len(results)} total")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── DEDUPLICATE ─────────────────────────────────────────────────────
def deduplicate(df):
    before = len(df)
    df = df.drop_duplicates(subset=["title"])
    df = df.drop_duplicates(subset=["text"])
    df = df[df["text"].str.split().str.len() >= 10]
    df["title_norm"] = df["title"].str.lower().str.strip()
    df = df.drop_duplicates(subset=["title_norm"])
    df = df.drop(columns=["title_norm"])
    df = df.reset_index(drop=True)
    after = len(df)
    print(f"Deduplication: {before} -> {after} rows ({before - after} removed)")
    return df

def fix_existing_csv(csv_path):
    df = pd.read_csv(csv_path)
    df["text"] = df["text"].fillna("").apply(truncate_60_words)
    return df[["title", "text", "label"]]

# ── MAIN ────────────────────────────────────────────────────────────
def build_full_dataset(existing_csv=None):
    all_results = []

    sources = [
        ("SNOPES",          scrape_snopes,          {"max_pages": 150}),
        ("POLITIFACT",      scrape_politifact,      {"max_pages": 100}),
        ("FACTCHECK",       scrape_factcheck,       {"max_pages": 50}),
        ("AFP",             scrape_afp,             {"max_pages": 50}),
        ("FULLFACT",        scrape_fullfact,        {"max_pages": 60}),
        ("LEADSTORIES",     scrape_leadstories,     {"max_pages": 70}),
        ("VISHVAS",         scrape_vishvas,         {"max_pages": 50}),
        ("BOOMLIVE",        scrape_boomlive,        {"max_pages": 50}),
        ("CHECKYOURFACT",   scrape_checkyourfact,   {"max_pages": 40}),
        ("LOGICALLY",       scrape_logically,       {"max_pages": 40}),
    ]

    for name, func, kwargs in sources:
        print(f"\n{'='*50}")
        print(f"SCRAPING {name}...")
        print(f"{'='*50}")
        try:
            data = func(**kwargs)
            all_results += data
            print(f"{name} done: {len(data)} articles")
        except Exception as e:
            print(f"{name} failed: {e}")
        
        # Save checkpoint after every source
        checkpoint_df = pd.DataFrame(all_results)
        checkpoint_df.to_csv("checkpoint_misleading.csv", index=False)
        print(f"Checkpoint saved: {len(all_results)} rows so far")

    new_df = pd.DataFrame(all_results)[["title", "text", "label"]]
    print(f"\nTotal freshly scraped: {len(new_df)}")

    # Merge with existing
    if existing_csv and os.path.exists(existing_csv):
        print(f"\nLoading existing: {existing_csv}")
        existing_df = fix_existing_csv(existing_csv)
        print(f"Existing rows: {len(existing_df)}")
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df

    print("\nDeduplicating...")
    final_df = deduplicate(combined_df)

    # Word count check
    final_df["wc"] = final_df["text"].str.split().str.len()
    print(f"\nWord count stats:\n{final_df['wc'].describe()}")
    final_df = final_df.drop(columns=["wc"])

    final_df.to_csv("misleading_7k.csv", index=False)
    print(f"\nDone! Saved {len(final_df)} rows to misleading_7k.csv")
    print(f"Label counts:\n{final_df['label'].value_counts()}")
    return final_df

# ── RUN ─────────────────────────────────────────────────────────────
df = build_full_dataset(existing_csv="snopes_misleading.csv")


SCRAPING SNOPES...
  [Snopes 1] Have Social Security checks been renamed 'federal benefit payments'? H
  [Snopes 2] Video doesn't show downed US pilot in Iranian custodyWritten by:Jordan
  [Snopes 3] White House posted and deleted video of Trump mocking SCOTUS justices,
  [Snopes 4] Robert Morris, former Trump faith adviser, released on probation after
  [Snopes 5] Did RFK Jr. announce Nutella is being pulled from US shelves?Written b
  [Snopes 6] Is Trump renaming Washington Monument after himself, plating it in gol
  [Snopes 7] Fake image of dad wearing MAGA hat blocking trans person from women's 
  [Snopes 8] Did Iran say it's starting to achieve US regime change following Pam B
  [Snopes 9] Image of Trump, Epstein with young female and children is fake. Here's
  [Snopes 10] Kamala Harris never said, 'Iran is a country, but we don't live there,
  [Snopes 11] Did Oprah try to bribe John Fetterman to vote against Trump's 'Big Bea
  [Snopes 12] Why Trump's birthright citizenship execu

In [1]:
# ═══════════════════════════════════════════════════════════════
# FRESH 20K MISLEADING — ALL NEW SOURCES NEVER USED BEFORE
# ═══════════════════════════════════════════════════════════════

# !pip install requests beautifulsoup4 pandas datasets -q

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os
from datasets import load_dataset

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept":          "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

all_results  = []
seen_titles  = set()
seen_texts   = set()

def safe_get(url, timeout=15):
    try:
        return requests.get(url, headers=HEADERS, timeout=timeout)
    except:
        return None

def truncate_60(text):
    if not text: return ""
    return " ".join(str(text).split()[:60])

def extract_text(soup, selectors):
    for sel in selectors:
        body = soup.select_one(sel)
        if body:
            return truncate_60(body.get_text(separator=" ", strip=True))
    return ""

def add(title, text, source):
    if not text or len(text.split()) < 8: return False
    t = title.lower().strip() if title else ""
    x = text.lower().strip()
    if t in seen_titles or x in seen_texts: return False
    all_results.append({"title": title, "text": text, "label": "misleading", "source": source})
    seen_titles.add(t)
    seen_texts.add(x)
    return True

def checkpoint():
    pd.DataFrame(all_results).to_csv("checkpoint_fresh_20k.csv", index=False)
    print(f"  >>> Checkpoint: {len(all_results)} rows saved")


# ════════════════════════════════════════════════════════════════
# PART A: HUGGINGFACE DATASETS (NEW ONES ONLY)
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("PART A: HUGGINGFACE DATASETS")
print("="*60)

# ── A1: CLEF2021 CheckThat ───────────────────────────────────────
try:
    print("\nLoading CLEF2021 CheckThat...")
    ds = load_dataset("tner/fin", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            text  = truncate_60(str(row.get("tokens", "")))
            title = text[:80]
            if add(title, text, "clef2021"): count += 1
    print(f"CLEF2021: {count} rows")
except Exception as e:
    print(f"CLEF2021 failed: {e}")

# ── A2: PUBHEALTH Dataset ────────────────────────────────────────
try:
    print("\nLoading PUBHEALTH dataset...")
    ds = load_dataset("health_fact", trust_remote_code=True)
    misleading_labels = {"mixture", "unproven", "false"}
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            if str(row.get("label", "")).lower() in misleading_labels:
                title = str(row.get("claim", ""))
                text  = truncate_60(
                    str(row.get("claim",        "")) + " " +
                    str(row.get("explanation",  ""))
                )
                if add(title, text, "pubhealth"): count += 1
    print(f"PUBHEALTH: {count} rows")
except Exception as e:
    print(f"PUBHEALTH failed: {e}")

# ── A3: COVID-19 Infodemic ───────────────────────────────────────
try:
    print("\nLoading COVID infodemic dataset...")
    ds = load_dataset("nyu-mll/multi_nli", trust_remote_code=True)
    count = 0
    for split in ["train", "validation_matched"]:
        if split not in ds: continue
        for row in ds[split]:
            if str(row.get("label", "")) == "1":  # neutral = misleading/uncertain
                title = str(row.get("premise", ""))[:80]
                text  = truncate_60(
                    str(row.get("premise",    "")) + " " +
                    str(row.get("hypothesis", ""))
                )
                if add(title, text, "multinli_neutral"): count += 1
                if count >= 3000: break
        if count >= 3000: break
    print(f"MultiNLI neutral: {count} rows")
except Exception as e:
    print(f"MultiNLI failed: {e}")

# ── A4: CONSTRAINT Dataset (COVID fake news) ─────────────────────
try:
    print("\nLoading CONSTRAINT dataset...")
    ds = load_dataset("Yaswanthkumar/COVID_FAKE_NEWS", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            title = str(row.get("tweet", ""))[:80]
            text  = truncate_60(str(row.get("tweet", "")))
            if add(title, text, "constraint_covid"): count += 1
    print(f"CONSTRAINT: {count} rows")
except Exception as e:
    print(f"CONSTRAINT failed: {e}")

# ── A5: ISOT Fake News ───────────────────────────────────────────
try:
    print("\nLoading ISOT dataset...")
    ds = load_dataset("GonzaloA/fake_news", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            title = str(row.get("title", ""))
            text  = truncate_60(str(row.get("text", "")))
            if add(title, text, "isot_fakenews"): count += 1
            if count >= 2000: break
        if count >= 2000: break
    print(f"ISOT: {count} rows")
except Exception as e:
    print(f"ISOT failed: {e}")

# ── A6: NELA-GT Dataset ──────────────────────────────────────────
try:
    print("\nLoading NELA-GT...")
    ds = load_dataset("NELA2019", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            if int(row.get("label", -1)) == 1:  # mixed reliability
                title = str(row.get("title", ""))
                text  = truncate_60(str(row.get("content", "")))
                if add(title, text, "nela_gt"): count += 1
    print(f"NELA-GT: {count} rows")
except Exception as e:
    print(f"NELA-GT failed: {e}")

# ── A7: SemEval 2019 RumourEval ──────────────────────────────────
try:
    print("\nLoading RumourEval...")
    ds = load_dataset("strombergnlp/rumoureval_2019", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            if str(row.get("label", "")).lower() in {"unverified", "query"}:
                title = str(row.get("text", ""))[:80]
                text  = truncate_60(str(row.get("text", "")))
                if add(title, text, "rumoureval"): count += 1
    print(f"RumourEval: {count} rows")
except Exception as e:
    print(f"RumourEval failed: {e}")

# ── A8: MediaEval Verifying Multimedia ──────────────────────────
try:
    print("\nLoading Pheme dataset...")
    ds = load_dataset("pheme", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            if str(row.get("veracity", "")).lower() == "unverified":
                title = str(row.get("text", ""))[:80]
                text  = truncate_60(str(row.get("text", "")))
                if add(title, text, "pheme"): count += 1
    print(f"PHEME: {count} rows")
except Exception as e:
    print(f"PHEME failed: {e}")

# ── A9: Fakeddit ─────────────────────────────────────────────────
try:
    print("\nLoading Fakeddit...")
    ds = load_dataset("Fakeddit", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            if str(row.get("2_way_label", "")) == "0":
                title = str(row.get("title", ""))
                text  = truncate_60(str(row.get("title", "")))
                if add(title, text, "fakeddit"): count += 1
                if count >= 2000: break
        if count >= 2000: break
    print(f"Fakeddit: {count} rows")
except Exception as e:
    print(f"Fakeddit failed: {e}")

# ── A10: TruthSeeker ─────────────────────────────────────────────
try:
    print("\nLoading TruthSeeker...")
    ds = load_dataset("IlyaGusev/truthseeker", trust_remote_code=True)
    count = 0
    for split in ds.keys():
        for row in ds[split]:
            title = str(row.get("statement", ""))[:80]
            text  = truncate_60(str(row.get("statement", "")) + " " + str(row.get("justification", "")))
            if add(title, text, "truthseeker"): count += 1
    print(f"TruthSeeker: {count} rows")
except Exception as e:
    print(f"TruthSeeker failed: {e}")

checkpoint()
print(f"\nAfter Part A: {len(all_results)} rows")


# ════════════════════════════════════════════════════════════════
# PART B: BRAND NEW SCRAPING SOURCES
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("PART B: NEW SCRAPING SOURCES")
print("="*60)

# ── B1: PolitiFact (Pants on Fire) ──────────────────────────────
def scrape_politifact_new(max_pages=80):
    results = []
    for rating in ["pants-fire", "false"]:
        for page in range(1, max_pages + 1):
            url = f"https://www.politifact.com/factchecks/list/?ruling={rating}&page={page}"
            r = safe_get(url)
            if not r: break
            soup = BeautifulSoup(r.text, "html.parser")
            items = soup.select("li.o-listicle__item")
            if not items:
                print(f"[PolitiFact-{rating}] Stopped at page {page}")
                break
            for item in items:
                link_tag = item.select_one("a.m-statement__quote")
                if not link_tag: continue
                title = link_tag.get_text(strip=True)
                href  = "https://www.politifact.com" + link_tag["href"]
                r2    = safe_get(href)
                if not r2: continue
                s2    = BeautifulSoup(r2.text, "html.parser")
                text  = extract_text(s2, ["article.m-textblock", "div.m-textblock", "main"])
                if add(title, text, f"politifact_{rating}"): 
                    results.append({"title": title, "text": text})
                    print(f"  [PolitiFact-{rating} {len(results)}] {title[:60]}")
                time.sleep(random.uniform(0.8, 1.5))
            time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B2: Africa Check ─────────────────────────────────────────────
def scrape_africacheck(max_pages=80):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://africacheck.org/fact-checks?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='teaser'], div[class*='card']")
        if not cards:
            print(f"[AfricaCheck] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3, [class*='title']")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            if href.startswith("/"): href = "https://africacheck.org" + href
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.field--body", "div.article-body", "div.entry-content", "main"])
            if add(title, text, "africacheck"):
                results.append({"title": title, "text": text})
                print(f"  [AfricaCheck {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[AfricaCheck] Page {page} -- {len(results)} new")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B3: Pesacheck ────────────────────────────────────────────────
def scrape_pesacheck(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://pesacheck.org/tagged/fact-check?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='post']")
        if not cards:
            print(f"[Pesacheck] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            if href.startswith("/"): href = "https://pesacheck.org" + href
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.section-content", "div.entry-content", "main"])
            if add(title, text, "pesacheck"):
                results.append({"title": title, "text": text})
                print(f"  [Pesacheck {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B4: Maldita.es (English) ─────────────────────────────────────
def scrape_maldita(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://maldita.es/malditobulo/noticias/?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='article']")
        if not cards:
            print(f"[Maldita] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            if href.startswith("/"): href = "https://maldita.es" + href
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.article-content", "div.entry-content", "main"])
            if add(title, text, "maldita"):
                results.append({"title": title, "text": text})
                print(f"  [Maldita {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B5: Pagella Politica ─────────────────────────────────────────
def scrape_pagella(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://pagellapolitica.it/fact-checking?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='card']")
        if not cards:
            print(f"[Pagella] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3, [class*='title']")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            if href.startswith("/"): href = "https://pagellapolitica.it" + href
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.article-body", "div.content", "main"])
            if add(title, text, "pagella"):
                results.append({"title": title, "text": text})
                print(f"  [Pagella {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B6: Science Feedback ─────────────────────────────────────────
def scrape_sciencefeedback(max_pages=50):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://sciencefeedback.co/fact-checks/?foogrid_page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='post']")
        if not cards:
            print(f"[ScienceFeedback] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.entry-content", "div.post-content", "main"])
            if add(title, text, "sciencefeedback"):
                results.append({"title": title, "text": text})
                print(f"  [ScienceFeedback {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B7: Climate Feedback ─────────────────────────────────────────
def scrape_climatefeedback(max_pages=40):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://climatefeedback.org/fact-checks/?foogrid_page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='post']")
        if not cards:
            print(f"[ClimateFeedback] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.entry-content", "div.article-body", "main"])
            if add(title, text, "climatefeedback"):
                results.append({"title": title, "text": text})
                print(f"  [ClimateFeedback {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B8: Snopes (FALSE category — new label) ──────────────────────
def scrape_snopes_false(max_pages=100):
    results = []
    base_url = "https://www.snopes.com/fact-check/?filter=false"
    for page in range(1, max_pages + 1):
        r = safe_get(f"{base_url}&pagenum={page}")
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("div.article_wrapper")
        if not cards:
            print(f"[Snopes-False] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a[href*='/fact-check/']")
            title_tag = card.select_one("div.article_text")
            if not link_tag or not title_tag: continue
            href  = link_tag.get("href", "")
            if href.startswith("/"): href = "https://www.snopes.com" + href
            title = title_tag.get_text(strip=True)
            r2    = safe_get(href)
            if not r2: continue
            s2    = BeautifulSoup(r2.text, "html.parser")
            text  = extract_text(s2, ["div.single-body", "div.article-text", "main"])
            if add(title, text, "snopes_false"):
                results.append({"title": title, "text": text})
                print(f"  [Snopes-False {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[Snopes-False] Page {page} -- {len(results)} new")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B9: MediaBiasFacts ───────────────────────────────────────────
def scrape_mediabias(max_pages=40):
    results = []
    for page in range(1, max_pages + 1):
        url = f"https://mediabiasfactcheck.com/fake-news/?page={page}"
        r = safe_get(url)
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("article, div[class*='entry']")
        if not cards:
            print(f"[MediaBias] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a")
            title_tag = card.select_one("h2, h3")
            if not link_tag or not title_tag: continue
            title = title_tag.get_text(strip=True)
            href  = link_tag.get("href", "")
            r2 = safe_get(href)
            if not r2: continue
            s2   = BeautifulSoup(r2.text, "html.parser")
            text = extract_text(s2, ["div.entry-content", "div.post-content", "main"])
            if add(title, text, "mediabias"):
                results.append({"title": title, "text": text})
                print(f"  [MediaBias {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── B10: Snopes (MOSTLY FALSE category) ─────────────────────────
def scrape_snopes_mostlyfalse(max_pages=80):
    results = []
    base_url = "https://www.snopes.com/fact-check/?filter=mostly-false"
    for page in range(1, max_pages + 1):
        r = safe_get(f"{base_url}&pagenum={page}")
        if not r: break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("div.article_wrapper")
        if not cards:
            print(f"[Snopes-MostlyFalse] Stopped at page {page}")
            break
        for card in cards:
            link_tag  = card.select_one("a[href*='/fact-check/']")
            title_tag = card.select_one("div.article_text")
            if not link_tag or not title_tag: continue
            href  = link_tag.get("href", "")
            if href.startswith("/"): href = "https://www.snopes.com" + href
            title = title_tag.get_text(strip=True)
            r2    = safe_get(href)
            if not r2: continue
            s2    = BeautifulSoup(r2.text, "html.parser")
            text  = extract_text(s2, ["div.single-body", "div.article-text", "main"])
            if add(title, text, "snopes_mostlyfalse"):
                results.append({"title": title, "text": text})
                print(f"  [Snopes-MostlyFalse {len(results)}] {title[:60]}")
            time.sleep(random.uniform(0.8, 1.5))
        print(f"[Snopes-MostlyFalse] Page {page} -- {len(results)} new")
        time.sleep(random.uniform(1.0, 2.0))
    return results

# ── RUN ALL SCRAPERS ─────────────────────────────────────────────
scrapers = [
    ("POLITIFACT NEW RATINGS", scrape_politifact_new,    {"max_pages": 80}),
    ("AFRICA CHECK",           scrape_africacheck,        {"max_pages": 80}),
    ("PESACHECK",              scrape_pesacheck,          {"max_pages": 50}),
    ("MALDITA",                scrape_maldita,            {"max_pages": 50}),
    ("PAGELLA POLITICA",       scrape_pagella,            {"max_pages": 50}),
    ("SCIENCE FEEDBACK",       scrape_sciencefeedback,    {"max_pages": 50}),
    ("CLIMATE FEEDBACK",       scrape_climatefeedback,    {"max_pages": 40}),
    ("SNOPES FALSE",           scrape_snopes_false,       {"max_pages": 100}),
    ("MEDIA BIAS",             scrape_mediabias,          {"max_pages": 40}),
    ("SNOPES MOSTLY FALSE",    scrape_snopes_mostlyfalse, {"max_pages": 80}),
]

for name, func, kwargs in scrapers:
    print(f"\n{'='*50}\nSCRAPING {name}...\n{'='*50}")
    try:
        func(**kwargs)
        print(f"{name} done. Total so far: {len(all_results)}")
    except Exception as e:
        print(f"{name} failed: {e}")
    checkpoint()


# ════════════════════════════════════════════════════════════════
# FINAL SAVE
# ════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("FINAL SAVE")
print("="*60)

final_df = pd.DataFrame(all_results)[["title", "text", "label"]]

# Final cleanup
final_df = final_df.drop_duplicates(subset=["title"])
final_df = final_df.drop_duplicates(subset=["text"])
final_df["title_norm"] = final_df["title"].str.lower().str.strip()
final_df = final_df.drop_duplicates(subset=["title_norm"])
final_df = final_df.drop(columns=["title_norm"])
final_df = final_df[final_df["text"].str.split().str.len() >= 8]
final_df = final_df.reset_index(drop=True)

print(f"\nFinal dataset: {len(final_df)} rows")
print(f"Word count:\n{final_df['text'].str.split().str.len().describe()}")
print(f"\nLabel distribution:\n{final_df['label'].value_counts()}")

final_df.to_csv("fresh_20k_misleading.csv", index=False)
print(f"\nSaved to fresh_20k_misleading.csv")
print(final_df.head())

d:\Trustcheck\trustcheck\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.3.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
d:\Trustcheck\trustcheck\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'tner/fin' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



PART A: HUGGINGFACE DATASETS

Loading CLEF2021 CheckThat...


d:\Trustcheck\trustcheck\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahmad\.cache\huggingface\hub\datasets--tner--fin. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'health_fact' isn't based on a loading scrip

CLEF2021 failed: Dataset scripts are no longer supported, but found fin.py

Loading PUBHEALTH dataset...


d:\Trustcheck\trustcheck\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ahmad\.cache\huggingface\hub\datasets--health_fact. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'nyu-mll/multi_nli' isn't based on a loadi

...

RumourEval failed: Dataset scripts are no longer supported, but found rumoureval_2019.py

Loading Pheme dataset...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Fakeddit' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


PHEME failed: Dataset 'pheme' doesn't exist on the Hub or cannot be accessed.

Loading Fakeddit...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'IlyaGusev/truthseeker' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Fakeddit failed: Dataset 'Fakeddit' doesn't exist on the Hub or cannot be accessed.

Loading TruthSeeker...
TruthSeeker failed: Dataset 'IlyaGusev/truthseeker' doesn't exist on the Hub or cannot be accessed.
  >>> Checkpoint: 5000 rows saved

After Part A: 5000 rows

PART B: NEW SCRAPING SOURCES

SCRAPING POLITIFACT NEW RATINGS...
POLITIFACT NEW RATINGS done. Total so far: 5000
  >>> Checkpoint: 5000 rows saved

SCRAPING AFRICA CHECK...
AFRICA CHECK done. Total so far: 5000
  >>> Checkpoint: 5000 rows saved

SCRAPING PESACHECK...
PESACHECK done. Total so far: 5000
  >>> Checkpoint: 5000 rows saved

SCRAPING MALDITA...
MALDITA done. Total so far: 5000
  >>> Checkpoint: 5000 rows saved

SCRAPING PAGELLA POLITICA...
[Pagella] Stopped at page 1
PAGELLA POLITICA done. Total so far: 5000
  >>> Checkpoint: 5000 rows saved

SCRAPING SCIENCE FEEDBACK...
SCIENCE FEEDBACK done. Total so far: 5000
  >>> Checkpoint: 5000 rows saved

SCRAPING CLIMATE FEEDBACK...
CLIMATE FEEDBACK done. Total so far: 